# Face Aging Progression (StarGAN) — Colab
تدريب وتوليد وتقييم موديل شيخوخة الوجه على UTKFace بحجم 456×456.

**أولًا:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1) إحضار الكود
ارفع مجلد المشروع `face-model` إلى Colab (أو ادفعه إلى GitHub واستنسخه). ثم عدّل المسار:

In [ ]:
import os
PROJECT = '/content/face-model'   # عدّل إن لزم
# مثال للاستنساخ من GitHub بدلًا من الرفع:
# !git clone https://github.com/<user>/<repo>.git /content/face-model
assert os.path.isdir(PROJECT), 'ارفع مجلد المشروع أو استنسخه أولًا'
os.chdir(PROJECT); print('cwd =', os.getcwd())

In [ ]:
# 2) المتطلبات (torch مثبّت مسبقًا على Colab)
!pip install -q pyyaml torchmetrics torch-fidelity lpips scipy tqdm
!pip install -q facenet-pytorch --no-deps   # بدون تبعيات حتى لا يخفّض torch

In [ ]:
# 3) ربط Google Drive لحفظ الأوزان والصور لكل إيبوك
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/face_aging_out'
os.makedirs(OUT, exist_ok=True); print('outputs ->', OUT)

## 4) تحميل UTKFace عبر Kaggle
ارفع ملف `kaggle.json` (من Account → Create New API Token).

In [ ]:
from google.colab import files
files.upload()   # اختر kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d jangedoo/utkface-new -p /content --unzip
# المجلد الناتج عادة /content/UTKFace أو /content/utkface_aligned_cropped/UTKFace
!ls /content | head

In [ ]:
DATA = '/content/UTKFace'   # عدّل حسب مسار فك الضغط
!python scripts/prepare_utkface.py --data-root $DATA

In [ ]:
# 5) تجربة سريعة للتأكد أن الأنبوب يعمل
!python scripts/train.py --config configs/debug.yaml --data-root $DATA --output-root $OUT

In [ ]:
# 6) التدريب الكامل 456×456 (طويل — استخدم --resume عند الانقطاع)
!python scripts/train.py --config configs/utkface_456.yaml --data-root $DATA --output-root $OUT

In [ ]:
# 7) عرض آخر صورة عيّنات (تتابع الأعمار)
import glob
from IPython.display import Image as IM, display
s = sorted(glob.glob(f'{OUT}/utkface_456/samples/epoch_*.png'))
if s: display(IM(filename=s[-1]))

In [ ]:
# 8) توليد شيخوخة تتابعية من صورة مدخلة
W = f'{OUT}/utkface_456/weights/best_G.pth'
!python scripts/generate.py --config configs/utkface_456.yaml --weights $W \
  --input /content/UTKFace --output /content/results --targets all --align
r = sorted(glob.glob('/content/results/*_aging_strip.png'))
if r: display(IM(filename=r[0]))

In [ ]:
# 9) التقييم بالمقاييس الخمسة (FID/SSIM/LPIPS/PSNR/CSIM)
!python scripts/evaluate.py --config configs/utkface_456.yaml --weights $W --split test --output $OUT/utkface_456/eval